In [1]:
import pandas as pd
import numpy as np
import os

# 1. --- LOAD DATABASE ---
filename = os.path.join('..', 'nba_modern_filtered_master.csv')
df = pd.read_csv(filename)

# Dynamically select numeric features, ignoring IDs and rank targets
exclude_cols = [
    'Player_Name', 'Era_Label', 'Archetype_Label', 'Era_ID', 'Archetype_ID',
    'Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank', 
    'Modern_Athletic_Raw_Rank', 'Modern_Yahoo_Raw_Rank', 'Modern_BR_Raw_Rank',
    'Modern_Consensus_Rank', 'Scratch_DL_Score', 'Scratch_DL_Rank'
]
features = [col for col in df.columns if col in df.select_dtypes(include=[np.number]).columns and col not in exclude_cols]

# 2. --- ERA-RELATIVE STANDARDIZATION ---
df_scaled = df.copy()

# FIX: Force all analytical columns to float64 so they can safely accept Z-score decimals
df_scaled[features] = df_scaled[features].astype(float)

for era in df['Era_Label'].unique():
    era_mask = df['Era_Label'] == era
    # Ensure there's more than one player in the era to avoid division by zero
    if era_mask.sum() > 1:
        for col in features:
            era_mean = df.loc[era_mask, col].mean()
            era_std = df.loc[era_mask, col].std()
            if era_std != 0:
                df_scaled.loc[era_mask, col] = (df.loc[era_mask, col] - era_mean) / era_std
            else:
                df_scaled.loc[era_mask, col] = 0.0

# Clean any residual missing entries after scaling if they exist
X = np.nan_to_num(df_scaled[features].values)

# Target setup (Invert ranks so higher = better)
if 'Modern_Consensus_Rank' in df.columns:
    y_raw = df['Modern_Consensus_Rank'].values
else:
    raw_cols = ['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']
    valid_raw = [c for c in raw_cols if c in df.columns]
    y_raw = df[valid_raw].mean(axis=1).values

# Ensure target ranks don't contain NaNs before building the vector
y_raw = pd.to_numeric(y_raw, errors='coerce')
y_raw = np.nan_to_num(y_raw, nan=np.nanmedian(y_raw))

y_target = (max(y_raw) + 1 - y_raw).reshape(-1, 1)

# 3. --- CUSTOM MODEL TRAINING (GRADIENT DESCENT) ---
np.random.seed(42)
m, n = X.shape

# Initialize weights and bias near zero
W = np.random.randn(n, 1) * 0.01
b = 0.0

learning_rate = 0.01
epochs = 1500

for epoch in range(epochs):
    # Forward Pass
    y_pred = np.dot(X, W) + b
    
    # Calculate Gradients
    error = y_pred - y_target
    dW = np.dot(X.T, error) / m
    db = np.sum(error) / m
    
    # Update Parameters
    W -= learning_rate * dW
    b -= learning_rate * db

# 4. --- FEATURE IMPORTANCE RANKING ---
# Extract absolute weight values
importances = np.abs(W.flatten())

# Pair features with their absolute learned weight
feature_rankings = sorted(zip(features, importances), key=lambda x: x[1], reverse=True)

print("⚖️ CUSTOM MODEL: ERA-ADJUSTED FEATURE IMPORTANCE")
print("=" * 60)
print(f"{'Rank':<5} | {'Statistical Factor':<25} | {'Weight Magnitude':<15}")
print("-" * 60)

for rank, (feat, imp) in enumerate(feature_rankings, 1):
    print(f"{rank:<5} | {feat:<25} | {imp:<15.4f}")

⚖️ CUSTOM MODEL: ERA-ADJUSTED FEATURE IMPORTANCE
Rank  | Statistical Factor        | Weight Magnitude
------------------------------------------------------------
1     | All_NBA_PS                | 6.9171         
2     | Championships             | 4.6608         
3     | Peak_5Yr_PER              | 4.6384         
4     | All_NBA_Teams             | 4.4411         
5     | Playoff_PPG               | 2.3219         
6     | All_Defensive_Teams       | 2.2571         
7     | Career_TS_Pct             | 1.7257         
8     | Playoff_BPM               | 1.5501         
9     | Playoff_Games             | 1.4994         
10    | Career_PPG                | 1.4769         
11    | Career_BPM                | 1.3921         
12    | Playoff_WS_per_48         | 1.2627         
13    | Career_PER                | 1.1666         
14    | Playoff_PER               | 1.1349         
15    | Career_WS_per_48          | 1.0470         
16    | Seasons_Played            | 0.5222         
17   

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

def run_basketball_analytics_pipeline(input_filename, output_filename):
    # 1. --- LOAD DATABASE ---
    if not os.path.exists(input_filename):
        input_filename = 'nba_modern_filtered_master.csv'
        if not os.path.exists(input_filename):
            print("❌ Error: Target CSV file could not be found.")
            return
            
    df = pd.read_csv(input_filename)
    print(f"📖 Loaded '{input_filename}' containing {len(df)} players.")

    # 2. --- SYSTEM FEATURE ENGINEERING ---
    # Dynamically engineer the top machine-learned density driver
    df['All_NBA_Teams'] = pd.to_numeric(df['All_NBA_Teams'], errors='coerce').fillna(0.0)
    df['Seasons_Played'] = pd.to_numeric(df['Seasons_Played'], errors='coerce').fillna(1.0)
    
    df['All_NBA_PS'] = df['All_NBA_Teams'] / df['Seasons_Played']

    # 3. --- MACHINE-LEARNED OPTIMAL WEIGHTS ---
    machine_weights = {
    'All_NBA_PS': 6.9171,
    'Championships': 4.6608,
    'Peak_5Yr_PER': 4.6384,
    'All_NBA_Teams': 4.4411,
    'Playoff_PPG': 2.3219,
    'All_Defensive_Teams': 2.2571,
    'Career_TS_Pct': 1.7257,
    'Playoff_BPM': 1.5501,
    'Playoff_Games': 1.4994,
    'Career_PPG': 1.4769,
    'Career_BPM': 1.3921,
    'Playoff_WS_per_48': 1.2627,
    'Career_PER': 1.1666,
    'Playoff_PER': 1.1349,
    'Career_WS_per_48': 1.0470,
    'Seasons_Played': 0.5222,
    'Finals_MVP': 0.4643,
    'Career_Games': 0.4314,
    'Playoff_Seasons': 0.3028
    }
    
    features = list(machine_weights.keys())
    valid_features = [col for col in features if col in df.columns]
    
    # 4. --- INTRA-ERA STANDARD SCALING (NEUTRALIZING ERAS) ---
    df_scaled = df.copy()
    df_scaled[valid_features] = df_scaled[valid_features].astype(float)
    
    for era in df['Era_Label'].unique():
        era_mask = df['Era_Label'] == era
        if era_mask.sum() > 1:
            for col in valid_features:
                era_mean = df.loc[era_mask, col].mean()
                era_std = df.loc[era_mask, col].std()
                
                if era_std != 0:
                    df_scaled.loc[era_mask, col] = (df.loc[era_mask, col] - era_mean) / era_std
                else:
                    df_scaled.loc[era_mask, col] = 0.0

    # 5. --- COMPUTE MACHINE-LEARNED INDEX MATRIX ---
    df['Custom_Index_Score'] = 0.0
    for col in valid_features:
        df['Custom_Index_Score'] += df_scaled[col] * machine_weights[col]

    # 6. --- GENERATE DENSE GAP-FREE RANKINGS ---
    df['Custom_Model_Rank'] = df['Custom_Index_Score'].rank(ascending=False, method='min').astype(int)
    
    # 7. --- COMPUTE DEVIATION AGAINST MEDIA CONSENSUS ---
    if 'Modern_Consensus_Rank' not in df.columns:
        raw_cols = ['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']
        valid_raw = [c for c in raw_cols if c in df.columns]
        if valid_raw:
            df['Modern_Consensus_Rank'] = df[valid_raw].mean(axis=1)
        else:
            df['Modern_Consensus_Rank'] = np.nan

    df_eval = df.dropna(subset=['Modern_Consensus_Rank'])

    if not df_eval.empty:
        df['Rank_Deviation'] = df['Custom_Model_Rank'] - df['Modern_Consensus_Rank']
        mae = df['Rank_Deviation'].abs().mean()
        r2_ranking = r2_score(df_eval['Modern_Consensus_Rank'], df_eval['Custom_Model_Rank'])
    else:
        df['Rank_Deviation'] = 0.0
        mae, r2_ranking = 0.0, 0.0

    df_final = df.sort_values(by='Custom_Model_Rank')

    # 8. --- EXPORT FILE ---
    df_final.to_csv(output_filename, index=False)
    print(f"💾 Success! Exported optimized machine results to '{output_filename}'")
    print(f"📊 Global Mean Absolute Error (MAE): {mae:.2f} spots")
    print(f"📉 Coefficient of Determination (R² Score): {r2_ranking:.4f}\n")

    # 9. --- DISPLAY LEADERBOARD BREAKDOWN ---
    print("🤖 ==================== MACHINE OPTIMIZED LIST ==================== 🤖")
    print(f"{'Model Rank':<11} | {'Player Name':<22} | {'Era':<12} | {'Net Shift':<10}")
    print("-" * 65)
    
    for _, row in df_final.head(20).iterrows():
        shift = row['Rank_Deviation']
        if shift < 0:
            shift_text = f"⬆️ {abs(shift):.1f}"
        elif shift > 0:
            shift_text = f"⬇️ {abs(shift):.1f}"
        else:
            shift_text = "🔄 0.0"
            
        print(f"{int(row['Custom_Model_Rank']):<11} | {row['Player_Name']:<22} | {row['Era_Label']:<12} | {shift_text:<10}")

if __name__ == '__main__':
    target_database = os.path.join('..', 'nba_modern_filtered_master.csv')
    output_database = 'nba_machine_optimized_results.csv'
    run_basketball_analytics_pipeline(target_database, output_database)

📖 Loaded '..\nba_modern_filtered_master.csv' containing 88 players.
💾 Success! Exported optimized machine results to 'nba_machine_optimized_results.csv'
📊 Global Mean Absolute Error (MAE): 9.33 spots
📉 Coefficient of Determination (R² Score): 0.7590

🤖 ==================== MACHINE OPTIMIZED LIST ==================== 🤖
Model Rank  | Player Name            | Era          | Net Shift 
-----------------------------------------------------------------
1           | LeBron James           | 2010-2019    | ⬆️ 1.0    
2           | Michael Jordan         | 1990-1999    | ⬇️ 1.0    
3           | Kobe Bryant            | 2000-2009    | ⬆️ 4.7    
4           | Kareem Abdul-Jabbar    | 1980-1989    | ⬇️ 1.0    
5           | Shaquille O'Neal       | 2000-2009    | ⬆️ 0.7    
6           | Tim Duncan             | 2000-2009    | ⬇️ 0.7    
7           | Karl Malone            | 1990-1999    | ⬆️ 7.7    
8           | Larry Bird             | 1980-1989    | ⬇️ 0.7    
9           | Kevin Durant   